# Temas 8 y 9 — Potencia en CA y Circuitos Trifásicos

**Teoría de Circuitos — ETSI, Universidad de Sevilla**

---

## Objetivos de aprendizaje

- Comprender la potencia instantánea, activa, reactiva y aparente en circuitos de corriente alterna
- Dominar el concepto de potencia compleja y el triángulo de potencias
- Aplicar la corrección del factor de potencia mediante condensadores
- Conocer el funcionamiento del transformador ideal y la reflexión de impedancias
- Analizar sistemas trifásicos equilibrados en conexión estrella (Y) y triángulo ($\Delta$)
- Calcular potencias trifásicas y resolver circuitos mediante el equivalente monofásico

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Arc
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA     = '#cb181d'   # rojo - rectas, límites
COLOR_PUNTO     = '#238b45'   # verde - puntos de operación
COLOR_N         = '#a6cee3'   # azul claro
COLOR_P         = '#b2df8a'   # verde claro
COLOR_FASE_A    = '#e41a1c'   # rojo  - fase A
COLOR_FASE_B    = '#377eb8'   # azul  - fase B
COLOR_FASE_C    = '#4daf4a'   # verde - fase C

print('Configuración lista.')

---

## 1. Introducción: ¿qué es la potencia en CA?

En corriente continua la potencia es simplemente $P = V \cdot I$. En **corriente alterna**, tensión y corriente son funciones senoidales que pueden estar **desfasadas** un ángulo $\varphi$. Esto hace que la potencia varíe con el tiempo y aparezcan conceptos nuevos: potencia **activa**, **reactiva** y **aparente**.

La idea clave es que la potencia instantánea $p(t) = u(t) \cdot i(t)$ tiene una **componente media constante** (que realiza trabajo útil) y una **componente oscilante** al doble de frecuencia (que intercambia energía con los elementos reactivos, $L$ y $C$).

---

## 2. Potencia instantánea

Sean

$$u(t) = \sqrt{2}\,U\,\cos(\omega t + \varphi_u), \qquad i(t) = \sqrt{2}\,I\,\cos(\omega t + \varphi_i)$$

donde $U$ e $I$ son valores eficaces y $\varphi = \varphi_u - \varphi_i$ es el desfase de la tensión respecto a la corriente.

La potencia instantánea es:

$$\boxed{p(t) = U I \cos\varphi + U I \cos(2\omega t + \varphi_u + \varphi_i)}$$

- **Primer término** $U I \cos\varphi$: valor medio constante $\to$ **potencia activa** $P$.
- **Segundo término**: oscilación a frecuencia $2\omega$ $\to$ intercambio de energía con $L$ y $C$.

Cuando $\varphi = 0$ (carga resistiva pura), $p(t) \ge 0$ siempre: toda la energía se disipa.
Cuando $\varphi \neq 0$, aparecen intervalos donde $p(t) < 0$: la carga **devuelve** energía a la fuente.

In [ ]:
# Gráfica de potencia instantánea: u(t), i(t), p(t)
fig, ax = plt.subplots(figsize=(12, 6))

f = 50  # Hz
omega = 2 * np.pi * f
U, I = 1.0, 0.8          # valores eficaces
phi_u, phi_i = 0, -np.pi/4   # desfase φ = φ_u - φ_i = π/4
phi = phi_u - phi_i

t = np.linspace(0, 2/f, 1000)  # 2 periodos
u = np.sqrt(2) * U * np.cos(omega * t + phi_u)
i_t = np.sqrt(2) * I * np.cos(omega * t + phi_i)
p = u * i_t

# Sombrear zonas positivas y negativas de p(t)
ax.fill_between(t * 1e3, p, 0, where=(p >= 0), alpha=0.25, color=COLOR_PUNTO,
                label=r'$p(t) > 0$ (energía entregada)')
ax.fill_between(t * 1e3, p, 0, where=(p < 0), alpha=0.25, color=COLOR_RECTA,
                label=r'$p(t) < 0$ (energía devuelta)')

ax.plot(t * 1e3, u, color=COLOR_PRINCIPAL, lw=2, label=r'$u(t)$')
ax.plot(t * 1e3, i_t, color=COLOR_FASE_C, lw=2, ls='--', label=r'$i(t)$')
ax.plot(t * 1e3, p, color=COLOR_RECTA, lw=2.5, label=r'$p(t) = u(t) \cdot i(t)$')

# Línea de potencia media
P_avg = U * I * np.cos(phi)
ax.axhline(P_avg, color='gray', ls=':', lw=1.5, label=f'$P = UI\\cos\\varphi = {P_avg:.2f}$ W')

ax.set_xlabel(r'Tiempo (ms)')
ax.set_ylabel(r'Amplitud (V, A, W)')
ax.set_title(r'Potencia instantánea con $\varphi = 45°$ (carga RL)')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. Potencia activa, reactiva, aparente y compleja

### 3.1 Potencia activa $P$

$$\boxed{P = U I \cos\varphi \quad [\text{W}]}$$

- Es el **valor medio** de $p(t)$.
- Representa la energía que se **transforma** en trabajo útil o calor.
- $\cos\varphi$ se denomina **factor de potencia** (fdp).

### 3.2 Potencia reactiva $Q$

$$\boxed{Q = U I \sin\varphi \quad [\text{VAr}]}$$

- Mide la energía que **oscila** entre fuente y carga sin realizar trabajo.
- $Q > 0$: carga **inductiva** (absorbe reactiva).
- $Q < 0$: carga **capacitiva** (genera reactiva).

### 3.3 Potencia aparente $S$

$$\boxed{S = U I \quad [\text{VA}]}$$

$$S^2 = P^2 + Q^2$$

- Es el producto de los valores eficaces. Dimensiona conductores y transformadores.

### 3.4 Potencia compleja $\mathbf{S}$

$$\boxed{\mathbf{S} = P + jQ = \mathbf{V} \cdot \mathbf{I}^*}$$

donde $\mathbf{I}^*$ es el **conjugado** del fasor corriente. Se cumple $|\mathbf{S}| = S$.

| Magnitud | Símbolo | Unidad | Fórmula |
|----------|---------|--------|---------|
| Potencia activa | $P$ | W | $UI\cos\varphi$ |
| Potencia reactiva | $Q$ | VAr | $UI\sin\varphi$ |
| Potencia aparente | $S$ | VA | $UI$ |
| Potencia compleja | $\mathbf{S}$ | VA | $P + jQ = \mathbf{V}\mathbf{I}^*$ |

---

## 4. Triángulo de potencias

El triángulo de potencias es la representación geométrica de $\mathbf{S} = P + jQ$:

- **Cateto horizontal**: $P$ (potencia activa)
- **Cateto vertical**: $Q$ (potencia reactiva)
- **Hipotenusa**: $S$ (potencia aparente)
- **Ángulo**: $\varphi = \arctan(Q/P)$

In [ ]:
# Triángulo de potencias
fig, ax = plt.subplots(figsize=(8, 5))

P, Q = 800, 600  # W, VAr
S = np.sqrt(P**2 + Q**2)
phi_deg = np.degrees(np.arctan2(Q, P))

# Dibujar triángulo
ax.arrow(0, 0, P, 0, head_width=15, head_length=20, fc=COLOR_PUNTO, ec=COLOR_PUNTO, lw=2)
ax.arrow(P, 0, 0, Q, head_width=15, head_length=20, fc=COLOR_PRINCIPAL, ec=COLOR_PRINCIPAL, lw=2)
ax.arrow(0, 0, P, Q, head_width=15, head_length=20, fc=COLOR_RECTA, ec=COLOR_RECTA, lw=2.5)

# Etiquetas
ax.text(P/2, -50, r'$P$ (W)', fontsize=14, ha='center', color=COLOR_PUNTO, fontweight='bold')
ax.text(P + 60, Q/2, r'$Q$ (VAr)', fontsize=14, ha='left', color=COLOR_PRINCIPAL, fontweight='bold')
ax.text(P/2 - 80, Q/2 + 30, r'$S = |\mathbf{S}|$ (VA)', fontsize=14, ha='center',
        color=COLOR_RECTA, fontweight='bold', rotation=np.degrees(np.arctan2(Q, P)))

# Arco para ángulo φ
arc_r = 120
theta = np.linspace(0, np.radians(phi_deg), 50)
ax.plot(arc_r * np.cos(theta), arc_r * np.sin(theta), color='gray', lw=1.5)
ax.text(arc_r * 0.75, 40, r'$\varphi$', fontsize=14, color='gray')

# Valores
ax.text(P + 60, Q + 40,
        f'$P = {P}$ W\n$Q = {Q}$ VAr\n$S = {S:.0f}$ VA\n'
        + f'$\\varphi = {phi_deg:.1f}°$\n$\\cos\\varphi = {np.cos(np.radians(phi_deg)):.2f}$',
        fontsize=12, bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))

ax.set_xlim(-50, 1100)
ax.set_ylim(-100, 800)
ax.set_aspect('equal')
ax.set_title('Triángulo de potencias', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. Potencia en R, L y C

| Elemento | $P$ | $Q$ | Interpretación |
|----------|-----|-----|---------------|
| **Resistencia** $R$ | $R I^2$ | $0$ | Solo consume activa (disipa calor) |
| **Inductancia** $L$ | $0$ | $\omega L I^2 > 0$ | **Absorbe** reactiva (almacena en campo magnético) |
| **Condensador** $C$ | $0$ | $-\dfrac{I^2}{\omega C} < 0$ | **Genera** reactiva (almacena en campo eléctrico) |

**Regla mnemotécnica:**
- Las bobinas son "consumidoras" de reactiva ($Q_L > 0$).
- Los condensadores son "productores" de reactiva ($Q_C < 0$).
- Las resistencias solo ven potencia activa.

---

## 6. Máxima transferencia de potencia en CA

Para un circuito con impedancia de Thévenin $\mathbf{Z}_{Th} = R_{Th} + jX_{Th}$, la máxima potencia se transfiere a la carga cuando:

$$\boxed{\mathbf{Z}_L = \mathbf{Z}_{Th}^* = R_{Th} - jX_{Th}}$$

Es decir, la impedancia de carga debe ser el **conjugado** de la impedancia de Thévenin. La potencia máxima entregada es:

$$\boxed{P_{\max} = \frac{|\mathbf{V}_{Th}|^2}{8\,R_{Th}}}$$

**Nota:** si la carga es puramente resistiva ($X_L = 0$), la condición se convierte en $R_L = |\mathbf{Z}_{Th}|$.

---

## 7. Corrección del factor de potencia

La mayoría de cargas industriales son **inductivas** ($\cos\varphi$ bajo, $Q > 0$). Para mejorar el factor de potencia se añade un **condensador en paralelo** que genera reactiva y compensa la de la bobina.

**Procedimiento:**

1. Calcular la potencia reactiva inicial: $Q_1 = P \tan\varphi_1$
2. Calcular la potencia reactiva deseada: $Q_2 = P \tan\varphi_2$
3. El condensador debe aportar: $Q_C = Q_1 - Q_2$
4. Valor del condensador:

$$\boxed{C = \frac{Q_C}{\omega V^2} = \frac{P(\tan\varphi_1 - \tan\varphi_2)}{\omega V^2}}$$

**Ventajas de corregir el fdp:**
- Menor corriente por la línea $\to$ menores pérdidas $R \cdot I^2$
- Menor caída de tensión
- Evitar penalizaciones de la compañía eléctrica

In [ ]:
# Triángulo de potencias: antes y después de la corrección
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

P = 1000  # W
phi1 = np.radians(60)  # cos(φ₁) = 0.5
phi2 = np.radians(18.2)  # cos(φ₂) ≈ 0.95

Q1 = P * np.tan(phi1)
Q2 = P * np.tan(phi2)
S1 = np.sqrt(P**2 + Q1**2)
S2 = np.sqrt(P**2 + Q2**2)
Qc = Q1 - Q2

for ax_i, (ax, Q, S, phi, title) in enumerate(zip(
    axes, [Q1, Q2], [S1, S2], [phi1, phi2],
    [r'Antes: $\cos\varphi_1 = 0.50$', r'Después: $\cos\varphi_2 = 0.95$'])):

    ax.arrow(0, 0, P, 0, head_width=15, head_length=20, fc=COLOR_PUNTO, ec=COLOR_PUNTO, lw=2)
    ax.arrow(P, 0, 0, Q, head_width=15, head_length=20, fc=COLOR_PRINCIPAL, ec=COLOR_PRINCIPAL, lw=2)
    ax.arrow(0, 0, P, Q, head_width=15, head_length=20, fc=COLOR_RECTA, ec=COLOR_RECTA, lw=2.5)

    ax.text(P/2, -80, f'$P = {P}$ W', fontsize=12, ha='center', color=COLOR_PUNTO, fontweight='bold')
    ax.text(P + 50, Q/2, f'$Q = {Q:.0f}$ VAr', fontsize=12, ha='left', color=COLOR_PRINCIPAL)
    ax.text(P/2 - 120, Q/2 + 40, f'$S = {S:.0f}$ VA', fontsize=12, ha='center', color=COLOR_RECTA)

    ax.set_xlim(-50, 1300)
    ax.set_ylim(-120, 2000)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if ax_i == 1:
        # Mostrar Q_C compensada
        ax.annotate('', xy=(P, Q2), xytext=(P, Q1),
                     arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
        ax.text(P + 50, (Q1 + Q2)/2, f'$Q_C = {Qc:.0f}$ VAr\n(condensador)',
                fontsize=11, color='purple')
        ax.arrow(P, 0, 0, Q1, head_width=0, head_length=0,
                 fc='none', ec='gray', lw=1, ls='--')

fig.suptitle('Corrección del factor de potencia', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 8. Transformador ideal

El transformador ideal se caracteriza por su **relación de transformación** $a = N_1/N_2$:

$$\boxed{\frac{V_1}{V_2} = \frac{N_1}{N_2} = a, \qquad I_1 \cdot N_1 = I_2 \cdot N_2}$$

### Reflexión de impedancias

Una impedancia $\mathbf{Z}_2$ conectada en el secundario se "ve" desde el primario como:

$$\boxed{\mathbf{Z}_1' = a^2 \cdot \mathbf{Z}_2 = \left(\frac{N_1}{N_2}\right)^2 \mathbf{Z}_2}$$

### Potencia

El transformador ideal **no disipa potencia**: $\mathbf{S}_1 = \mathbf{S}_2$.

| Magnitud | Relación |
|----------|---------|
| Tensiones | $V_1 = a \cdot V_2$ |
| Corrientes | $I_1 = I_2 / a$ |
| Impedancias | $Z_1' = a^2 \cdot Z_2$ |
| Potencia | $S_1 = S_2$ |

In [ ]:
# Diagrama del transformador ideal
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('Transformador ideal', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

# Primario
d += elm.SourceSin().up().label(r'$\mathbf{V}_1$', loc='left').length(4)
src_top = d.here
d += elm.Line().right().length(2)
d += elm.Inductor2(loops=4).down().label(r'$N_1$', loc='left').length(4)
d += elm.Line().left().length(2)

# Líneas de acoplamiento (núcleo)
d += elm.Line().right().at(src_top).length(2)
core_x = d.here[0] + 1.0

# Secundario
d += elm.Line().right().at((core_x + 1.5, src_top[1])).length(2)
sec_top = d.here
d += elm.Inductor2(loops=4).down().label(r'$N_2$', loc='right').length(4)
sec_bot = d.here
d += elm.Line().right().length(2)
d += elm.Resistor().up().label(r'$\mathbf{Z}_2$', loc='right').length(4)
d += elm.Line().left().length(2)

d.draw()
plt.tight_layout()
plt.show()

---

## 9. Sistemas trifásicos: generalidades

Un sistema trifásico equilibrado consta de **tres fuentes senoidales** de igual amplitud, desfasadas $120°$ entre sí. En **secuencia directa** (la más habitual):

$$\boxed{\mathbf{V}_a = V_f \angle 0°, \quad \mathbf{V}_b = V_f \angle{-120°}, \quad \mathbf{V}_c = V_f \angle{+120°}}$$

donde $V_f$ es el valor eficaz de la tensión de fase.

**Propiedad fundamental:** la suma de las tres tensiones (y corrientes) es siempre cero:

$$\mathbf{V}_a + \mathbf{V}_b + \mathbf{V}_c = 0$$

In [ ]:
# Tensiones trifásicas en el dominio del tiempo
fig, ax = plt.subplots(figsize=(12, 6))

f = 50
omega = 2 * np.pi * f
t = np.linspace(0, 2/f, 1000)

Vm = np.sqrt(2) * 230  # tensión de pico (230 V eficaz)
va = Vm * np.cos(omega * t)
vb = Vm * np.cos(omega * t - 2*np.pi/3)
vc = Vm * np.cos(omega * t + 2*np.pi/3)

ax.plot(t * 1e3, va, color=COLOR_FASE_A, lw=2.5, label=r'$v_a(t)$ — Fase A')
ax.plot(t * 1e3, vb, color=COLOR_FASE_B, lw=2.5, label=r'$v_b(t)$ — Fase B')
ax.plot(t * 1e3, vc, color=COLOR_FASE_C, lw=2.5, label=r'$v_c(t)$ — Fase C')

# Línea cero
ax.axhline(0, color='gray', lw=0.8)

ax.set_xlabel(r'Tiempo (ms)')
ax.set_ylabel(r'Tensión (V)')
ax.set_title(r'Sistema trifásico equilibrado — secuencia directa ($f = 50$ Hz, $V_f = 230$ V)')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 10. Conexión estrella (Y)

En conexión **estrella**, las tres fuentes (o cargas) comparten un punto neutro $N$. Se definen:

- **Tensión de fase** $V_f$: tensión entre cada terminal y el neutro.
- **Tensión de línea** $V_L$: tensión entre dos terminales.

$$\boxed{V_L = \sqrt{3}\,V_f, \qquad \mathbf{V}_{AB} = \sqrt{3}\,\mathbf{V}_{AN}\,e^{j30°}}$$

$$\boxed{I_L = I_f}$$

La tensión de línea es $\sqrt{3}$ veces la de fase y está **adelantada $30°$**.

---

## 11. Conexión triángulo ($\Delta$)

En conexión **triángulo**, no hay neutro; las cargas se conectan entre líneas.

$$\boxed{V_L = V_f}$$

$$\boxed{I_L = \sqrt{3}\,I_f, \qquad \mathbf{I}_L = \sqrt{3}\,\mathbf{I}_f\,e^{-j30°}}$$

La corriente de línea es $\sqrt{3}$ veces la de fase y está **retrasada $30°$**.

### Comparación Y vs $\Delta$

| Magnitud | Estrella (Y) | Triángulo ($\Delta$) |
|----------|-------------|---------------------|
| $V_L$ vs $V_f$ | $V_L = \sqrt{3}\,V_f$ | $V_L = V_f$ |
| $I_L$ vs $I_f$ | $I_L = I_f$ | $I_L = \sqrt{3}\,I_f$ |
| Neutro | Sí (opcional) | No |
| $\mathbf{Z}_f$ (Y $\to$ $\Delta$) | $\mathbf{Z}_\Delta = 3\,\mathbf{Z}_Y$ | $\mathbf{Z}_Y = \mathbf{Z}_\Delta / 3$ |

---

## 12. Circuito monofásico equivalente

Para un sistema trifásico **equilibrado**, basta resolver **una sola fase** y multiplicar por 3 para obtener la potencia total:

$$\boxed{P_{1\varphi} = V_f \cdot I_f \cdot \cos\varphi, \qquad P_{3\varphi} = 3\,P_{1\varphi}}$$

**Procedimiento:**

1. Convertir cargas en $\Delta$ a su equivalente Y: $\mathbf{Z}_Y = \mathbf{Z}_\Delta / 3$
2. Dibujar el circuito monofásico (una fase + neutro)
3. Resolver por ley de Ohm: $\mathbf{I}_f = \mathbf{V}_f / \mathbf{Z}_{total}$
4. Calcular potencias de una fase y multiplicar por 3

**Error frecuente:** usar $V_L$ donde debería ir $V_f$. Recordar siempre: en el monofásico equivalente se trabaja con tensiones **de fase**.

---

## 13. Potencia trifásica

En un sistema trifásico equilibrado, las fórmulas de potencia en función de magnitudes de **línea** son:

$$\boxed{P = \sqrt{3}\,V_L\,I_L\,\cos\varphi}$$

$$\boxed{Q = \sqrt{3}\,V_L\,I_L\,\sin\varphi}$$

$$\boxed{S = \sqrt{3}\,V_L\,I_L}$$

donde $\varphi$ es el ángulo de la **impedancia de fase**, no el ángulo entre tensión e intensidad de línea.

| Fórmula | Con magnitudes de fase | Con magnitudes de línea |
|---------|----------------------|------------------------|
| $P$ | $3\,V_f I_f \cos\varphi$ | $\sqrt{3}\,V_L I_L \cos\varphi$ |
| $Q$ | $3\,V_f I_f \sin\varphi$ | $\sqrt{3}\,V_L I_L \sin\varphi$ |
| $S$ | $3\,V_f I_f$ | $\sqrt{3}\,V_L I_L$ |

In [ ]:
# Diagrama fasorial: tensiones y corrientes en Y y Δ
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Estrella (Y): fasores de tensión de fase y línea ---
ax = axes[0]
ax.set_title(r'Conexión estrella (Y): $V_L = \sqrt{3} V_f \angle 30°$',
             fontsize=12, fontweight='bold')

# Tensiones de fase
Vf = 1.0
angles_f = [0, -120, 120]
labels_f = [r'$\mathbf{V}_{AN}$', r'$\mathbf{V}_{BN}$', r'$\mathbf{V}_{CN}$']
colors_f = [COLOR_FASE_A, COLOR_FASE_B, COLOR_FASE_C]

for ang, lbl, col in zip(angles_f, labels_f, colors_f):
    rad = np.radians(ang)
    ax.annotate('', xy=(Vf*np.cos(rad), Vf*np.sin(rad)), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(1.12*np.cos(rad), 1.12*np.sin(rad), lbl, fontsize=12,
            ha='center', va='center', color=col, fontweight='bold')

# Tensiones de línea (√3 Vf, adelantadas 30°)
VL = np.sqrt(3) * Vf
angles_L = [30, -90, 150]
labels_L = [r'$\mathbf{V}_{AB}$', r'$\mathbf{V}_{BC}$', r'$\mathbf{V}_{CA}$']

for ang, lbl in zip(angles_L, labels_L):
    rad = np.radians(ang)
    ax.annotate('', xy=(VL*np.cos(rad)*0.6, VL*np.sin(rad)*0.6), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls='--'))
    ax.text(VL*0.68*np.cos(rad), VL*0.68*np.sin(rad), lbl, fontsize=10,
            ha='center', va='center', color='gray')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Triángulo (Δ): fasores de corriente de fase y línea ---
ax = axes[1]
ax.set_title(r'Conexión triángulo ($\Delta$): $I_L = \sqrt{3} I_f \angle -30°$',
             fontsize=12, fontweight='bold')

If_ = 1.0
angles_if = [0, -120, 120]
labels_if = [r'$\mathbf{I}_{AB}$', r'$\mathbf{I}_{BC}$', r'$\mathbf{I}_{CA}$']

for ang, lbl, col in zip(angles_if, labels_if, colors_f):
    rad = np.radians(ang)
    ax.annotate('', xy=(If_*np.cos(rad), If_*np.sin(rad)), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(1.12*np.cos(rad), 1.12*np.sin(rad), lbl, fontsize=12,
            ha='center', va='center', color=col, fontweight='bold')

IL = np.sqrt(3) * If_
angles_iL = [-30, -150, 90]
labels_iL = [r'$\mathbf{I}_A$', r'$\mathbf{I}_B$', r'$\mathbf{I}_C$']

for ang, lbl in zip(angles_iL, labels_iL):
    rad = np.radians(ang)
    ax.annotate('', xy=(IL*np.cos(rad)*0.6, IL*np.sin(rad)*0.6), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls='--'))
    ax.text(IL*0.68*np.cos(rad), IL*0.68*np.sin(rad), lbl, fontsize=10,
            ha='center', va='center', color='gray')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 14. Metodología de resolución

### Problemas de potencia en CA (monofásico)

**Paso 1:** Identificar $\mathbf{V}$ e $\mathbf{I}$ (fasores) o bien $V$, $I$ y $\varphi$.

**Paso 2:** Calcular la potencia compleja: $\mathbf{S} = \mathbf{V} \cdot \mathbf{I}^*$.

**Paso 3:** Extraer $P = \text{Re}(\mathbf{S})$, $Q = \text{Im}(\mathbf{S})$, $S = |\mathbf{S}|$.

**Paso 4:** Factor de potencia: $\text{fdp} = \cos\varphi = P/S$.

### Problemas de corrección del factor de potencia

**Paso 1:** Calcular $Q_1 = P\tan\varphi_1$ con el fdp original.

**Paso 2:** Calcular $Q_2 = P\tan\varphi_2$ con el fdp deseado.

**Paso 3:** $Q_C = Q_1 - Q_2$, luego $C = Q_C / (\omega V^2)$.

### Problemas trifásicos equilibrados

**Paso 1:** Identificar conexión de la fuente (Y o $\Delta$) y de la carga (Y o $\Delta$).

**Paso 2:** Convertir todo a Y si es necesario: $\mathbf{Z}_Y = \mathbf{Z}_\Delta / 3$.

**Paso 3:** Dibujar el circuito monofásico equivalente.

**Paso 4:** Resolver: $\mathbf{I}_f = \mathbf{V}_f / \mathbf{Z}_{total}$.

**Paso 5:** Calcular potencias: $P_{3\varphi} = 3\,V_f\,I_f\,\cos\varphi = \sqrt{3}\,V_L\,I_L\,\cos\varphi$.

---

## 15. Ejemplos resueltos

#### Ejercicio resuelto 1: Cálculo de P, Q, S a partir de fasores

**Datos:** $\mathbf{V} = 230\angle 0°$ V, $\mathbf{I} = 10\angle{-36.87°}$ A.

**Paso 1:** Calcular $\mathbf{S} = \mathbf{V} \cdot \mathbf{I}^*$

$$\mathbf{I}^* = 10\angle{+36.87°}\;\text{A}$$

$$\mathbf{S} = 230\angle 0° \times 10\angle{36.87°} = 2300\angle{36.87°}\;\text{VA}$$

**Paso 2:** Extraer componentes

$$P = S\cos\varphi = 2300 \times \cos(36.87°) = 2300 \times 0.8 = 1840\;\text{W}$$

$$Q = S\sin\varphi = 2300 \times \sin(36.87°) = 2300 \times 0.6 = 1380\;\text{VAr}$$

**Paso 3:** Resultados

$$\boxed{P = 1840\;\text{W}, \quad Q = 1380\;\text{VAr}, \quad S = 2300\;\text{VA}, \quad \cos\varphi = 0.8\;\text{(inductivo)}}$$

$Q > 0 \to$ carga inductiva. El factor de potencia es $0.8$ inductivo.

In [ ]:
# Diagrama para el ejercicio 1: circuito monofásico con carga RL
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title('Ejercicio 1: Carga RL monofásica', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

d += elm.SourceSin().up().label(r'$230\angle 0°$ V', loc='left').length(4)
top = d.here
d += elm.Line().right().length(2)
d += elm.Resistor().right().label(r'$R$', loc='top')
d += elm.Inductor2().down().label(r'$L$', loc='right').length(4)
d += elm.Line().left().length(5.5)

# Etiqueta de corriente
d += elm.Line().at(top).right().length(0.5)

d.draw()
plt.tight_layout()
plt.show()

#### Ejercicio resuelto 2: Corrección del factor de potencia

**Datos:** Carga de $P = 5\;\text{kW}$, $\cos\varphi_1 = 0.6$ inductivo, $V = 230\;\text{V}$, $f = 50\;\text{Hz}$. Se desea corregir a $\cos\varphi_2 = 0.95$ inductivo.

**Paso 1:** Ángulos

$$\varphi_1 = \arccos(0.6) = 53.13°, \qquad \varphi_2 = \arccos(0.95) = 18.19°$$

**Paso 2:** Potencias reactivas

$$Q_1 = P\tan\varphi_1 = 5000 \times \tan(53.13°) = 5000 \times 1.333 = 6667\;\text{VAr}$$

$$Q_2 = P\tan\varphi_2 = 5000 \times \tan(18.19°) = 5000 \times 0.3287 = 1644\;\text{VAr}$$

**Paso 3:** Reactiva del condensador

$$Q_C = Q_1 - Q_2 = 6667 - 1644 = 5023\;\text{VAr}$$

**Paso 4:** Capacidad

$$C = \frac{Q_C}{\omega V^2} = \frac{5023}{2\pi \times 50 \times 230^2} = \frac{5023}{16\,619\,025} = 302.3\;\mu\text{F}$$

$$\boxed{C \approx 302\;\mu\text{F}}$$

#### Ejercicio resuelto 3: Transformador ideal

**Datos:** Transformador con $N_1/N_2 = 10$, tensión primaria $V_1 = 2300\;\text{V}$, carga en secundario $\mathbf{Z}_2 = 3 + j4\;\Omega$.

**Paso 1:** Tensión en el secundario

$$V_2 = \frac{V_1}{a} = \frac{2300}{10} = 230\;\text{V}$$

**Paso 2:** Corriente en el secundario

$$I_2 = \frac{V_2}{|\mathbf{Z}_2|} = \frac{230}{\sqrt{3^2 + 4^2}} = \frac{230}{5} = 46\;\text{A}$$

**Paso 3:** Corriente en el primario

$$I_1 = \frac{I_2}{a} = \frac{46}{10} = 4.6\;\text{A}$$

**Paso 4:** Impedancia vista desde el primario

$$\mathbf{Z}_1' = a^2 \cdot \mathbf{Z}_2 = 100 \times (3 + j4) = 300 + j400\;\Omega$$

**Paso 5:** Potencia

$$S = V_1 \cdot I_1 = 2300 \times 4.6 = 10\,580\;\text{VA}$$

$$\varphi = \arctan(4/3) = 53.13°, \quad \cos\varphi = 0.6$$

$$\boxed{P = S\cos\varphi = 6348\;\text{W}, \quad Q = S\sin\varphi = 8464\;\text{VAr}}$$

#### Ejercicio resuelto 4: Sistema Y-Y equilibrado

**Datos:** Fuente trifásica equilibrada $V_L = 400\;\text{V}$ (línea), carga en estrella $\mathbf{Z}_Y = 10 + j8\;\Omega$ por fase, secuencia directa.

**Paso 1:** Tensión de fase

$$V_f = \frac{V_L}{\sqrt{3}} = \frac{400}{\sqrt{3}} = 230.9\;\text{V}$$

**Paso 2:** Corriente de fase (= corriente de línea en Y)

$$I_f = \frac{V_f}{|\mathbf{Z}_Y|} = \frac{230.9}{\sqrt{10^2 + 8^2}} = \frac{230.9}{12.81} = 18.03\;\text{A}$$

**Paso 3:** Ángulo de la impedancia

$$\varphi = \arctan\left(\frac{8}{10}\right) = 38.66°, \quad \cos\varphi = 0.781$$

**Paso 4:** Potencias trifásicas

$$P = \sqrt{3} \times 400 \times 18.03 \times \cos(38.66°) = \sqrt{3} \times 400 \times 18.03 \times 0.781 = 9756\;\text{W}$$

$$Q = \sqrt{3} \times 400 \times 18.03 \times \sin(38.66°) = \sqrt{3} \times 400 \times 18.03 \times 0.625 = 7805\;\text{VAr}$$

$$S = \sqrt{3} \times 400 \times 18.03 = 12\,492\;\text{VA}$$

$$\boxed{I_L = 18.03\;\text{A}, \quad P = 9756\;\text{W}, \quad Q = 7805\;\text{VAr}, \quad S = 12\,492\;\text{VA}}$$

In [ ]:
# Diagrama Y-Y equilibrado
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title('Sistema Y-Y equilibrado', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

# Fuente fase A
d += elm.SourceSin().up().label(r'$\mathbf{V}_{AN}$', loc='left').length(3)
src_a_top = d.here
d += elm.Line().right().label(r'$\mathbf{I}_A$', loc='top').length(5)
load_a_top = d.here
d += elm.Resistor().down().label(r'$\mathbf{Z}_Y$', loc='right').length(1.5)
d += elm.Inductor2().down().length(1.5)
load_n = d.here
d += elm.Line().left().length(5)
src_n = d.here

# Neutro label
d += elm.Dot().at(src_n).label('N', loc='left')
d += elm.Dot().at(load_n).label("N'", loc='right')

# Neutro wire
d += elm.Line().at(src_n).right().length(5).linestyle('--').color('gray')

d.draw()
plt.tight_layout()
plt.show()

#### Ejercicio resuelto 5: Sistema Y-$\Delta$ equilibrado

**Datos:** Fuente trifásica equilibrada $V_L = 400\;\text{V}$, carga en triángulo $\mathbf{Z}_\Delta = 30 + j40\;\Omega$.

**Paso 1:** Convertir carga a estrella equivalente

$$\mathbf{Z}_Y = \frac{\mathbf{Z}_\Delta}{3} = \frac{30 + j40}{3} = 10 + j13.33\;\Omega$$

**Paso 2:** Tensión de fase de la fuente

$$V_f = \frac{400}{\sqrt{3}} = 230.9\;\text{V}$$

**Paso 3:** Corriente de línea

$$I_L = \frac{V_f}{|\mathbf{Z}_Y|} = \frac{230.9}{\sqrt{10^2 + 13.33^2}} = \frac{230.9}{16.67} = 13.85\;\text{A}$$

**Paso 4:** Corriente de fase en la carga (en $\Delta$)

$$I_f = \frac{I_L}{\sqrt{3}} = \frac{13.85}{\sqrt{3}} = 8.0\;\text{A}$$

**Paso 5:** Ángulo y potencias

$$\varphi = \arctan\left(\frac{40}{30}\right) = 53.13°, \quad \cos\varphi = 0.6$$

$$P = \sqrt{3} \times 400 \times 13.85 \times 0.6 = 5756\;\text{W}$$

$$Q = \sqrt{3} \times 400 \times 13.85 \times 0.8 = 7675\;\text{VAr}$$

$$S = \sqrt{3} \times 400 \times 13.85 = 9594\;\text{VA}$$

$$\boxed{I_L = 13.85\;\text{A}, \quad I_f = 8.0\;\text{A}, \quad P = 5756\;\text{W}, \quad Q = 7675\;\text{VAr}}$$

In [ ]:
# Diagrama Y-Δ
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title(r'Sistema Y-$\Delta$ equilibrado', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

# Fuente Y (simplificada)
d += elm.SourceSin().up().label(r'$\mathbf{V}_f$', loc='left').length(3)
top = d.here
d += elm.Line().right().label(r'$\mathbf{I}_A$', loc='top').length(4)
A = d.here

# Carga Δ: tres impedancias entre líneas
d += elm.Resistor().down().label(r'$\mathbf{Z}_\Delta$', loc='right').length(3)
B = d.here
d += elm.Line().left().length(4)
bot = d.here

# Línea intermedia (fase C)
d += elm.Line().at(A).right().length(2)
d += elm.Resistor().down().label(r'$\mathbf{Z}_\Delta$', loc='right').length(3)
C = d.here
d += elm.Line().left().length(2)

# Tercera impedancia (entre B y C)
d += elm.Resistor().at(B).right().label(r'$\mathbf{Z}_\Delta$', loc='bottom').length(2)

d.draw()
plt.tight_layout()
plt.show()

---

## 16. Catálogo completo de ejercicios: todos los patrones

| # | Tipo | Ecuación clave | Dificultad |
|---|------|---------------|-----------|
| 1 | Cálculo de $P$, $Q$, $S$ desde fasores | $\mathbf{S} = \mathbf{V}\mathbf{I}^*$ | Baja |
| 2 | Triángulo de potencias | $S^2 = P^2 + Q^2$ | Baja |
| 3 | Corrección del fdp | $C = Q_C / (\omega V^2)$ | Media |
| 4 | Máxima transferencia en CA | $\mathbf{Z}_L = \mathbf{Z}_{Th}^*$ | Media |
| 5 | Transformador: relación de tensiones y corrientes | $V_1/V_2 = a$, $I_1 a = I_2$ | Baja |
| 6 | Reflexión de impedancias | $\mathbf{Z}' = a^2 \mathbf{Z}$ | Media |
| 7 | Sistema Y-Y equilibrado | $I_L = V_f / Z_Y$ | Media |
| 8 | Sistema Y-$\Delta$ equilibrado | $\mathbf{Z}_Y = \mathbf{Z}_\Delta / 3$ | Media |
| 9 | Sistema $\Delta$-$\Delta$ equilibrado | $I_f = V_L / Z_\Delta$ | Media |
| 10 | Conversión Y $\leftrightarrow$ $\Delta$ | $\mathbf{Z}_\Delta = 3\mathbf{Z}_Y$ | Baja |
| 11 | Potencia trifásica total | $P = \sqrt{3} V_L I_L \cos\varphi$ | Baja |
| 12 | Equivalente monofásico | Resolver 1 fase, $\times 3$ | Media |

### 16.1 Tipo 1: Cálculo de $P$, $Q$, $S$ desde fasores

$$\boxed{\mathbf{S} = \mathbf{V} \cdot \mathbf{I}^* = P + jQ}$$

**Procedimiento:**
1. Conjugar el fasor corriente: $\mathbf{I}^* = I\angle{-\varphi_i}$
2. Multiplicar: $\mathbf{S} = V\angle\varphi_v \times I\angle{-\varphi_i} = VI\angle(\varphi_v - \varphi_i)$
3. $P = \text{Re}(\mathbf{S})$, $Q = \text{Im}(\mathbf{S})$, $S = |\mathbf{S}|$

**Efecto de los parámetros:**
- Si **$\varphi$ aumenta** $\to$ $\cos\varphi$ disminuye $\to$ $P$ disminuye $\to$ $Q$ aumenta (más reactiva)
- Si **$|\mathbf{Z}|$ aumenta** $\to$ $I$ disminuye $\to$ $S$ disminuye (menos potencia total)

#### Ejercicio: Tipo 1

**Datos:** $\mathbf{V} = 120\angle 30°$ V, $\mathbf{I} = 5\angle{-15°}$ A.

$$\mathbf{S} = 120\angle 30° \times 5\angle 15° = 600\angle 45°\;\text{VA}$$

$$P = 600\cos(45°) = 424.3\;\text{W}$$

$$Q = 600\sin(45°) = 424.3\;\text{VAr}$$

$$\boxed{P = 424.3\;\text{W}, \quad Q = 424.3\;\text{VAr}, \quad S = 600\;\text{VA}, \quad \cos\varphi = 0.707}$$

### 16.2 Tipo 2: Triángulo de potencias

$$\boxed{S^2 = P^2 + Q^2, \quad \varphi = \arctan(Q/P), \quad \text{fdp} = \cos\varphi}$$

**Truco para el examen:** si te dan dos de las tres magnitudes ($P$, $Q$, $S$), la tercera sale directamente del triángulo.

#### Ejercicio: Tipo 2

**Datos:** Una carga consume $P = 3\;\text{kW}$ y $Q = 4\;\text{kVAr}$ (inductivo).

$$S = \sqrt{P^2 + Q^2} = \sqrt{3000^2 + 4000^2} = \sqrt{25 \times 10^6} = 5000\;\text{VA}$$

$$\varphi = \arctan(4000/3000) = 53.13°$$

$$\boxed{S = 5\;\text{kVA}, \quad \cos\varphi = 0.6\;\text{(inductivo)}}$$

### 16.3 Tipo 3: Corrección del factor de potencia

$$\boxed{C = \frac{P(\tan\varphi_1 - \tan\varphi_2)}{\omega V^2}}$$

**Efecto de los parámetros:**
- Si **el fdp deseado es mayor** $\to$ $\tan\varphi_2$ menor $\to$ $Q_C$ mayor $\to$ $C$ mayor
- Si **la tensión $V$ es mayor** $\to$ menos capacidad necesaria (relación inversa con $V^2$)
- Si **la frecuencia aumenta** $\to$ menos capacidad (relación inversa con $\omega$)

#### Ejercicio: Tipo 3

**Datos:** $P = 10\;\text{kW}$, $\cos\varphi_1 = 0.7$ inductivo, $V = 400\;\text{V}$, $f = 50\;\text{Hz}$. Corregir a $\cos\varphi_2 = 0.9$.

$$\varphi_1 = \arccos(0.7) = 45.57°, \quad \varphi_2 = \arccos(0.9) = 25.84°$$

$$Q_C = 10000(\tan 45.57° - \tan 25.84°) = 10000(1.020 - 0.484) = 5360\;\text{VAr}$$

$$C = \frac{5360}{2\pi \times 50 \times 400^2} = \frac{5360}{50\,265\,482} = 106.6\;\mu\text{F}$$

$$\boxed{C \approx 107\;\mu\text{F}}$$

In [ ]:
# Diagrama: carga RL con condensador de corrección en paralelo
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('Tipo 3: Corrección del factor de potencia', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

d += elm.SourceSin().up().label(r'$\mathbf{V}$', loc='left').length(4)
top = d.here
d += elm.Line().right().length(2)
j1 = d.here

# Rama de la carga (R + jX)
d += elm.Line().right().at(j1).length(3)
d += elm.Resistor().down().label(r'$R$', loc='right').length(2)
d += elm.Inductor2().down().label(r'$jX_L$', loc='right').length(2)
d += elm.Line().left().length(3)
j2 = d.here

# Condensador en paralelo
d += elm.Capacitor().at(j1).down().label(r'$C$', loc='right').length(4)

# Cerrar circuito
d += elm.Line().left().at(j2).length(2)

d.draw()
plt.tight_layout()
plt.show()

### 16.4 Tipo 4: Máxima transferencia de potencia en CA

$$\boxed{\mathbf{Z}_L = \mathbf{Z}_{Th}^* = R_{Th} - jX_{Th}}$$

$$\boxed{P_{\max} = \frac{V_{Th}^2}{8\,R_{Th}}}$$

**Efecto de los parámetros:**
- Si **$R_{Th}$ aumenta** $\to$ $P_{\max}$ disminuye (más pérdidas internas)
- La parte reactiva se **cancela** exactamente: $X_L + X_{Th} = 0$

#### Ejercicio: Tipo 4

**Datos:** $\mathbf{V}_{Th} = 100\angle 0°\;\text{V}$, $\mathbf{Z}_{Th} = 20 + j15\;\Omega$.

$$\mathbf{Z}_L = 20 - j15\;\Omega$$

$$P_{\max} = \frac{100^2}{8 \times 20} = \frac{10000}{160} = 62.5\;\text{W}$$

$$\boxed{\mathbf{Z}_L = 20 - j15\;\Omega, \quad P_{\max} = 62.5\;\text{W}}$$

### 16.5 Tipo 5: Transformador — relación de tensiones y corrientes

$$\boxed{V_2 = \frac{V_1}{a}, \qquad I_1 = \frac{I_2}{a}, \qquad a = \frac{N_1}{N_2}}$$

#### Ejercicio: Tipo 5

**Datos:** Transformador con $a = N_1/N_2 = 5$, $V_1 = 1150\;\text{V}$, carga $R_L = 10\;\Omega$ en el secundario.

$$V_2 = \frac{1150}{5} = 230\;\text{V}$$

$$I_2 = \frac{V_2}{R_L} = \frac{230}{10} = 23\;\text{A}$$

$$I_1 = \frac{I_2}{a} = \frac{23}{5} = 4.6\;\text{A}$$

$$\boxed{V_2 = 230\;\text{V}, \quad I_2 = 23\;\text{A}, \quad I_1 = 4.6\;\text{A}}$$

### 16.6 Tipo 6: Reflexión de impedancias

$$\boxed{\mathbf{Z}' = a^2 \cdot \mathbf{Z}_2}$$

**Efecto de los parámetros:**
- Si **$a > 1$** (reductor) $\to$ la impedancia vista desde el primario es **mayor** que la real
- Si **$a < 1$** (elevador) $\to$ la impedancia vista es **menor**

#### Ejercicio: Tipo 6

**Datos:** $a = 4$, $\mathbf{Z}_2 = 5 + j3\;\Omega$.

$$\mathbf{Z}' = 4^2 \times (5 + j3) = 16 \times (5 + j3) = 80 + j48\;\Omega$$

$$\boxed{\mathbf{Z}' = 80 + j48\;\Omega}$$

### 16.7 Tipo 7: Sistema Y-Y equilibrado

$$\boxed{V_f = \frac{V_L}{\sqrt{3}}, \qquad I_L = I_f = \frac{V_f}{\mathbf{Z}_Y}}$$

**Procedimiento:**
1. Obtener $V_f$ a partir de $V_L$.
2. Calcular $I_f = V_f / |\mathbf{Z}_Y|$.
3. En Y: $I_L = I_f$.
4. Potencia: $P_{3\varphi} = 3 R I_f^2$ o $P_{3\varphi} = \sqrt{3} V_L I_L \cos\varphi$.

#### Ejercicio: Tipo 7

**Datos:** $V_L = 380\;\text{V}$, $\mathbf{Z}_Y = 6 + j8\;\Omega$.

$$V_f = \frac{380}{\sqrt{3}} = 219.4\;\text{V}$$

$$I_L = I_f = \frac{219.4}{\sqrt{6^2 + 8^2}} = \frac{219.4}{10} = 21.94\;\text{A}$$

$$\varphi = \arctan(8/6) = 53.13°, \quad \cos\varphi = 0.6$$

$$P = \sqrt{3} \times 380 \times 21.94 \times 0.6 = 8664\;\text{W}$$

$$\boxed{I_L = 21.94\;\text{A}, \quad P = 8664\;\text{W}}$$

### 16.8 Tipo 8: Sistema Y-$\Delta$ equilibrado

$$\boxed{\mathbf{Z}_Y = \frac{\mathbf{Z}_\Delta}{3}}$$

**Procedimiento:**
1. Convertir $\mathbf{Z}_\Delta$ a $\mathbf{Z}_Y$.
2. Resolver como Y-Y.
3. Las corrientes de línea son las del equivalente Y.
4. Las corrientes de fase en la carga $\Delta$: $I_{f\Delta} = I_L / \sqrt{3}$.

#### Ejercicio: Tipo 8

**Datos:** $V_L = 400\;\text{V}$, $\mathbf{Z}_\Delta = 24 + j18\;\Omega$.

$$\mathbf{Z}_Y = \frac{24 + j18}{3} = 8 + j6\;\Omega, \quad |\mathbf{Z}_Y| = 10\;\Omega$$

$$V_f = \frac{400}{\sqrt{3}} = 230.9\;\text{V}, \quad I_L = \frac{230.9}{10} = 23.09\;\text{A}$$

$$I_{f\Delta} = \frac{23.09}{\sqrt{3}} = 13.33\;\text{A}$$

$$\varphi = \arctan(6/8) = 36.87°, \quad \cos\varphi = 0.8$$

$$P = \sqrt{3} \times 400 \times 23.09 \times 0.8 = 12\,800\;\text{W}$$

$$\boxed{I_L = 23.09\;\text{A}, \quad I_{f\Delta} = 13.33\;\text{A}, \quad P = 12.8\;\text{kW}}$$

### 16.9 Tipo 9: Sistema $\Delta$-$\Delta$ equilibrado

En un sistema $\Delta$-$\Delta$, la tensión de fase en la carga es directamente $V_L$:

$$\boxed{I_{f\Delta} = \frac{V_L}{|\mathbf{Z}_\Delta|}, \qquad I_L = \sqrt{3}\,I_{f\Delta}}$$

#### Ejercicio: Tipo 9

**Datos:** $V_L = 400\;\text{V}$, $\mathbf{Z}_\Delta = 15 + j20\;\Omega$.

$$|\mathbf{Z}_\Delta| = \sqrt{15^2 + 20^2} = 25\;\Omega$$

$$I_{f\Delta} = \frac{400}{25} = 16\;\text{A}, \quad I_L = \sqrt{3} \times 16 = 27.71\;\text{A}$$

$$\varphi = \arctan(20/15) = 53.13°, \quad \cos\varphi = 0.6$$

$$P = \sqrt{3} \times 400 \times 27.71 \times 0.6 = 11\,520\;\text{W}$$

$$\boxed{I_{f\Delta} = 16\;\text{A}, \quad I_L = 27.71\;\text{A}, \quad P = 11.52\;\text{kW}}$$

### 16.10 Tipo 10: Conversión Y $\leftrightarrow$ $\Delta$

$$\boxed{\mathbf{Z}_\Delta = 3\,\mathbf{Z}_Y \qquad \Leftrightarrow \qquad \mathbf{Z}_Y = \frac{\mathbf{Z}_\Delta}{3}}$$

Estas relaciones son válidas **solo para cargas equilibradas** (las tres impedancias iguales).

#### Ejercicio: Tipo 10

**Datos:** Carga en $\Delta$ con $\mathbf{Z}_\Delta = 30\angle 60°\;\Omega$. Obtener el equivalente Y.

$$\mathbf{Z}_Y = \frac{30\angle 60°}{3} = 10\angle 60° = 5 + j8.66\;\Omega$$

$$\boxed{\mathbf{Z}_Y = 10\angle 60°\;\Omega = 5 + j8.66\;\Omega}$$

### 16.11 Tipo 11: Potencia trifásica total

$$\boxed{P = \sqrt{3}\,V_L\,I_L\,\cos\varphi = 3\,V_f\,I_f\,\cos\varphi = 3\,R\,I_f^2}$$

**Error frecuente:** confundir $V_L$ con $V_f$ en la fórmula. Recordar:
- Fórmula con $\sqrt{3}$ $\to$ usa magnitudes de **línea**
- Fórmula con $3$ $\to$ usa magnitudes de **fase**

#### Ejercicio: Tipo 11

**Datos:** $V_L = 400\;\text{V}$, $I_L = 10\;\text{A}$, $\cos\varphi = 0.85$ inductivo.

$$P = \sqrt{3} \times 400 \times 10 \times 0.85 = 5888\;\text{W}$$

$$Q = \sqrt{3} \times 400 \times 10 \times \sin(\arccos 0.85) = \sqrt{3} \times 400 \times 10 \times 0.527 = 3650\;\text{VAr}$$

$$S = \sqrt{3} \times 400 \times 10 = 6928\;\text{VA}$$

$$\boxed{P = 5888\;\text{W}, \quad Q = 3650\;\text{VAr}, \quad S = 6928\;\text{VA}}$$

### 16.12 Tipo 12: Equivalente monofásico

**Procedimiento:**
1. Si la carga es $\Delta$, convertir a Y: $\mathbf{Z}_Y = \mathbf{Z}_\Delta/3$.
2. $V_f = V_L / \sqrt{3}$.
3. $\mathbf{Z}_{total} = \mathbf{Z}_{linea} + \mathbf{Z}_Y$ (incluir impedancia de línea si la hay).
4. $\mathbf{I}_f = \mathbf{V}_f / \mathbf{Z}_{total}$.
5. $P_{1\varphi} = V_f I_f \cos\varphi$, luego $P_{3\varphi} = 3 P_{1\varphi}$.

#### Ejercicio: Tipo 12

**Datos:** $V_L = 400\;\text{V}$, impedancia de línea $\mathbf{Z}_{lin} = 1 + j2\;\Omega$, carga $\mathbf{Z}_Y = 10 + j8\;\Omega$.

$$V_f = \frac{400}{\sqrt{3}} = 230.9\;\text{V}$$

$$\mathbf{Z}_{total} = (1 + j2) + (10 + j8) = 11 + j10\;\Omega$$

$$|\mathbf{Z}_{total}| = \sqrt{11^2 + 10^2} = 14.87\;\Omega$$

$$I_f = \frac{230.9}{14.87} = 15.53\;\text{A}$$

Potencia **en la carga** (solo $\mathbf{Z}_Y$):

$$P_{carga} = 3 \times R_Y \times I_f^2 = 3 \times 10 \times 15.53^2 = 7234\;\text{W}$$

Pérdidas en la línea:

$$P_{linea} = 3 \times R_{lin} \times I_f^2 = 3 \times 1 \times 15.53^2 = 723\;\text{W}$$

$$\boxed{I_f = 15.53\;\text{A}, \quad P_{carga} = 7234\;\text{W}, \quad P_{linea} = 723\;\text{W}}$$

In [ ]:
# Diagrama: equivalente monofásico con impedancia de línea
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title('Tipo 12: Circuito monofásico equivalente', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

d += elm.SourceSin().up().label(r'$\mathbf{V}_f$', loc='left').length(3)
top = d.here
d += elm.Line().right().length(1)
d += elm.Resistor().right().label(r'$R_{lin}$', loc='top')
d += elm.Inductor2().right().label(r'$jX_{lin}$', loc='top')
d += elm.Resistor().down().label(r'$R_Y$', loc='right').length(1.5)
d += elm.Inductor2().down().label(r'$jX_Y$', loc='right').length(1.5)
d += elm.Line().left().length(7)

d.draw()
plt.tight_layout()
plt.show()

---

## 17. Resumen y tabla de fórmulas clave

### Potencia en CA (monofásico)

| Fórmula | Descripción |
|---------|-------------|
| $\boxed{P = UI\cos\varphi}$ | **Potencia activa** [W] |
| $\boxed{Q = UI\sin\varphi}$ | **Potencia reactiva** [VAr] |
| $\boxed{S = UI}$ | **Potencia aparente** [VA] |
| $\boxed{\mathbf{S} = \mathbf{V}\mathbf{I}^* = P + jQ}$ | **Potencia compleja** |
| $S^2 = P^2 + Q^2$ | Relación del triángulo de potencias |
| $\text{fdp} = \cos\varphi = P/S$ | Factor de potencia |
| $C = Q_C / (\omega V^2)$ | Condensador para corrección del fdp |
| $\mathbf{Z}_L = \mathbf{Z}_{Th}^*$ | Máxima transferencia de potencia |

### Transformador ideal

| Fórmula | Descripción |
|---------|-------------|
| $V_1/V_2 = N_1/N_2 = a$ | Relación de transformación |
| $I_1 \cdot N_1 = I_2 \cdot N_2$ | Conservación de amperios-vuelta |
| $\mathbf{Z}' = a^2 \cdot \mathbf{Z}_2$ | Reflexión de impedancias |

### Sistemas trifásicos

| Fórmula | Descripción |
|---------|-------------|
| $V_L = \sqrt{3}\,V_f$ | Estrella: tensión de línea vs fase |
| $I_L = \sqrt{3}\,I_f$ | Triángulo: corriente de línea vs fase |
| $\mathbf{Z}_\Delta = 3\,\mathbf{Z}_Y$ | Conversión Y $\leftrightarrow$ $\Delta$ |
| $\boxed{P = \sqrt{3}\,V_L I_L \cos\varphi}$ | **Potencia activa trifásica** |
| $\boxed{Q = \sqrt{3}\,V_L I_L \sin\varphi}$ | **Potencia reactiva trifásica** |
| $\boxed{S = \sqrt{3}\,V_L I_L}$ | **Potencia aparente trifásica** |

### Errores frecuentes

- Usar $V_L$ donde debe ir $V_f$ (o viceversa) en las fórmulas de potencia.
- Olvidar conjugar $\mathbf{I}$ al calcular $\mathbf{S} = \mathbf{V}\mathbf{I}^*$.
- Confundir el signo de $Q$: positivo = inductivo, negativo = capacitivo.
- En la corrección del fdp, usar $\omega = 2\pi f$, no $f$ directamente.
- En el transformador, $I_1 = I_2/a$ (la corriente se divide por $a$, no se multiplica).